## Setup: Mount Drive & Install Dependencies

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# install required libraries
import regex as re
import emoji
import pandas as pd
import ast
import os
import shutil
from datasets import Dataset

## Load HC3 Data

In [ ]:
ds = pd.read_csv("/content/drive/MyDrive/colab_hc3_bundle/HC3-Dataset-Samples/hc3_unified_10000_seed42.csv")

## Stylistic Cleanup Functions

In [ ]:
EM_DASH_CHARS = ["\u2014", "\u2013"]  # em dash and en dash in unicode (since ChatGPT and other LLMs use both for em dashes)

def remove_em_dashes(text):
    for c in EM_DASH_CHARS:
        text = re.sub(rf'\s+{re.escape(c)}\s+', ', ', text)  # converts "hello - world" -> "hello word"
        text = re.sub(rf'{re.escape(c)}\s*', ', ', text)      # converts "hello-world" -> "hello word"
    text = re.sub(r',\s*,', ',', text)    # double commas
    text = re.sub(r'^\s*,\s*', '', text)  # leading comma
    text = re.sub(r'\s{2,}', ' ', text)   # extra whitespace
    return text.strip()

In [ ]:
def remove_emojis(text):
  # covers emoticons, pictographs, transport icons, flags, supplemental/misc symbols
    EMOJI_PATTERN = re.compile(
        "[\U0001F600-\U0001F64F\U0001F300-\U0001F5FF"
        "\U0001F680-\U0001F6FF\U0001F1E0-\U0001F1FF"
        "\U00002700-\U000027BF\U0001F900-\U0001F9FF"
        "\U00002600-\U000026FF]+",
        flags=re.UNICODE
    )
    # remove using list of unicode emoji patterns
    text = EMOJI_PATTERN.sub('', text)
    text = re.sub(r'\s{2,}', ' ', text)
    return text.strip()

In [ ]:
def remove_formatting(text: str) -> str:
    # Headers: ## Heading → "Heading."
    def header_to_prose(m):
        heading = m.group(1).strip()
        if heading and heading[-1] not in '.!?':
            heading += '.'
        return heading
    text = re.sub(r'^#{1,6}\s+(.+)', header_to_prose, text, flags=re.MULTILINE)

    # Remove horizontal rules (--- or ===)
    text = re.sub(r'^\s*[-*=]{3,}\s*$', '', text, flags=re.MULTILINE)

    # Remove bold text only — leave * and _ alone to avoid breaking math/code
    text = re.sub(r'\*\*([^*\n]+?)\*\*', r'\1', text)
    text = re.sub(r'__([^_\n]+?)__', r'\1', text)

    # Convert bullets/numbered lists into prose (aka normal text)
    lines = text.split('\n')
    prose_chunks = []
    list_items = []

    def flush_list():
        if list_items:
            prose_chunks.append('. '.join(list_items) + '.')
            list_items.clear()

    for line in lines:
        stripped = line.strip()
        bullet  = re.match(r'^[-*\u2022]\s+(.+)', stripped)
        ordered = re.match(r'^\d+\.\s+(.+)', stripped)
        if bullet:
            list_items.append(bullet.group(1).strip())
        elif ordered:
            list_items.append(ordered.group(1).strip())
        else:
            flush_list()
            if stripped:
                prose_chunks.append(stripped)

    flush_list()
    text = ' '.join(prose_chunks)
    text = re.sub(r'\s{2,}', ' ', text)
    return text.strip()

In [ ]:
# run all the stylic cleanup functions required (removing em dashes, removing emojis, and replacing/removing markdown formatting)
def run_stylistic_cleanup(text):
    if not isinstance(text, str):
        return text  # safety guard so that we never return None
    text = remove_em_dashes(text)
    text = remove_emojis(text)
    text = remove_formatting(text)
    return text

## Apply Cleanup to HC3 Dataset

In [ ]:
def apply_and_track(example: dict) -> dict:
    """
    Apply stylistic cleanup to chatgpt_answers and track
    whether each row was modified or not.
    """
    original_answers  = example["chatgpt_answers"]
    perturbed_answers = [run_stylistic_cleanup(ans) for ans in original_answers]

    was_changed = any(
        orig != perturbed
        for orig, perturbed in zip(original_answers, perturbed_answers)
    )

    return {
        "chatgpt_answers_stylistic": perturbed_answers,
        "stylistic_changed":         was_changed,
    }


# Parse lists from CSV back into actual Python lists.
def safe_parse(val):
    if isinstance(val, list):
        return val
    try:
        return ast.literal_eval(val)
    except:
        return [val]


# Parse chatgpt_answers column from string → list
ds["chatgpt_answers"] = ds["chatgpt_answers"].apply(safe_parse)

# Convert to HuggingFace Dataset (final form) and apply cleanup
small_dataset = Dataset.from_pandas(ds)
small_dataset = small_dataset.map(apply_and_track)

In [ ]:
# Summary stats on the new dataset after applying changes
total   = len(small_dataset["chatgpt_answers"])
changed = sum(small_dataset["stylistic_changed"])
print(f"Rows changed:   {changed} / {total} ({changed/total*100:.1f}%)")
print(f"Rows unchanged: {total - changed} / {total} ({(total-changed)/total*100:.1f}%)")

# Preview changed rows
output_df = small_dataset.to_pandas()
# output_df[output_df['stylistic_changed'] == True].head(5)

## Save CSV to Drive

In [ ]:
# Save CSV to Drive (safe location, separate from git repo)
csv_path = "/content/drive/MyDrive/hc3_sample_10K_stylistic_cleanup.csv"
output_df.to_csv(csv_path, index=False)
print(f"Saved to {csv_path}")